In [3]:
import duckdb
import pandas as pd

with duckdb.connect("../data/creditlens.duckdb") as con:

    # Finding 1: Dual stress segment
    print(con.execute("""
        SELECT
            dual_stress_flag,
            COUNT(*)                            AS loans,
            ROUND(AVG(default_flag) * 100, 2)  AS default_rate_pct
        FROM model_features
        GROUP BY dual_stress_flag
        ORDER BY dual_stress_flag
    """).fetchdf())

    # Finding 2: Payment ratio by default status
    print(con.execute("""
        SELECT
            default_flag,
            ROUND(AVG(avg_payment_ratio_m6), 3)    AS avg_pay_ratio,
            ROUND(AVG(missed_payment_flag), 3)     AS pct_missed,
            ROUND(AVG(deteriorating_flag), 3)      AS pct_deteriorating,
            COUNT(*)                               AS loans
        FROM model_features
        GROUP BY default_flag
        ORDER BY default_flag
    """).fetchdf())

    # Finding 3: DTI + utilisation interaction
    print(con.execute("""
        SELECT
            high_dti_flag,
            high_util_flag,
            COUNT(*)                            AS loans,
            ROUND(AVG(default_flag) * 100, 2)  AS default_rate_pct
        FROM model_features
        GROUP BY high_dti_flag, high_util_flag
        ORDER BY high_dti_flag, high_util_flag
    """).fetchdf())

   dual_stress_flag  loans  default_rate_pct
0                 0  47697             11.78
1                 1   1645             16.66
   default_flag  avg_pay_ratio  pct_missed  pct_deteriorating  loans
0             0          1.232       0.090              0.090  43448
1             1          1.047       0.094              0.086   5894
   high_dti_flag  high_util_flag  loans  default_rate_pct
0              0               0  34306             10.78
1              0               1   7523             14.42
2              1               0   5868             14.28
3              1               1   1645             16.66
